# Lab 0 · Preparation
MICAI 2026 · Reliable Enterprise Agents · do this before the session (about 15 minutes)

1. Create a free API key at **console.groq.com** (API Keys, Create API Key).
2. **In Colab:** click the key icon in the left sidebar, add a secret named `GROQ_API_KEY`, and enable notebook access.
   **Locally with uv:** put `GROQ_API_KEY=...` in a `.env` file inside `labs/` (or `export GROQ_API_KEY=...`) before `uv run jupyter lab`.
3. Run every cell. If the last one prints a stock number, you are ready.

Use your own key: the free tier allows about 30 requests and 8,000 tokens per minute per key.

In [ ]:
import json
import os
import random
import time
import uuid
import hashlib
from pathlib import Path

# Locally (uv) everything is already installed. In Colab we install the
# packages and download the data files from the course repository.
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import subprocess
    import urllib.request
    subprocess.run(["pip", "install", "-q", "openai==3.41.0"], check=True)
    Path("data").mkdir(exist_ok=True)
    repo = "https://raw.githubusercontent.com/RamsesCamas/Enterprise-Agents-MICAI-2026/main/labs/data/"
    for name in ["products", "customers", "invoices", "documents"]:
        if not Path(f"data/{name}.jsonl").exists():
            urllib.request.urlretrieve(repo + f"{name}.jsonl", f"data/{name}.jsonl")

print("Data files:", sorted(p.name for p in Path("data").glob("*.jsonl")))

In [ ]:
! uv add openai

In [ ]:
from openai import OpenAI


def _load_dotenv():
    """Read a .env file into os.environ when running locally (not in Colab).

    Looks for a .env next to the notebook and in parent folders. Values
    already set in the real environment win, so `export GROQ_API_KEY=...`
    still takes precedence over the file.
    """
    for folder in [Path.cwd(), *Path.cwd().parents]:
        env_file = folder / ".env"
        if env_file.exists():
            for line in env_file.read_text().splitlines():
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                key, value = line.split("=", 1)
                key = key.strip()
                value = value.strip().strip('"').strip("'")
                os.environ.setdefault(key, value)
            break


def get_secret(name):
    if IN_COLAB:
        from google.colab import userdata
        return userdata.get(name)
    # Locally: use the real environment first, then fall back to a .env file.
    if name not in os.environ:
        _load_dotenv()
    return os.environ.get(name)


client = OpenAI(
    base_url=os.environ.get("LLM_BASE_URL", "https://api.groq.com/openai/v1"),
    api_key=get_secret("GROQ_API_KEY") or "missing-key",
    max_retries=4,
)
MODEL_NAME = os.environ.get("LLM_MODEL", "openai/gpt-oss-120b")


def ask_model(messages, tools=None):
    """Send the conversation to the model. Return its reply as a plain dict."""
    if tools:
        response = client.chat.completions.create(
            model=MODEL_NAME, messages=messages, tools=tools, temperature=0)
    else:
        response = client.chat.completions.create(
            model=MODEL_NAME, messages=messages, temperature=0)

    message = response.choices[0].message
    reply = {"role": "assistant", "content": message.content}
    if message.tool_calls:
        reply["tool_calls"] = []
        for call in message.tool_calls:
            reply["tool_calls"].append({
                "id": call.id,
                "type": "function",
                "function": {"name": call.function.name, "arguments": call.function.arguments},
            })
    reply["tokens"] = response.usage.total_tokens
    return reply

In [ ]:
print(get_secret("GROQ_API_KEY"))

## 1. One plain call

In [ ]:
reply = ask_model([{"role": "user", "content": "In one sentence: what is an idempotency key?"}])
print(reply["content"])
print("tokens used:", reply["tokens"])

## 2. The company we will work with
Acme's data lives in `data/*.jsonl`. The ERP class below loads it.

In [ ]:
def read_jsonl(path):
    rows = []
    with open(path) as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows


class ERPError(Exception):
    def __init__(self, code, message, retryable=False, retry_after=None):
        super().__init__(message)
        self.code = code
        self.message = message
        self.retryable = retryable
        self.retry_after = retry_after


class AcmeERP:
    """A small in-memory ERP loaded from data/*.jsonl.

    It behaves like a real backend: errors with codes, idempotency keys on
    writes, and outages you can simulate with fail_next().
    """

    def __init__(self, data_dir="data"):
        self.products = {}
        for row in read_jsonl(f"{data_dir}/products.jsonl"):
            self.products[row["sku"]] = row
        self.customers = {}
        for row in read_jsonl(f"{data_dir}/customers.jsonl"):
            self.customers[row["id"]] = row
        self.invoices = {}
        for row in read_jsonl(f"{data_dir}/invoices.jsonl"):
            self.invoices[row["id"]] = row
        self.movements = []
        self.refunds = []
        self.saved_replies = {}      # idempotency key -> reply already sent
        self.pending_failures = []

    def fail_next(self, kind, times=1):
        """kind: 'unavailable', 'rate_limited', 'timeout' or 'lost_response'."""
        for _ in range(times):
            self.pending_failures.append(kind)

    def _check_failures(self):
        """Raise a simulated outage. Return True if the reply must be 'lost'."""
        if not self.pending_failures:
            return False
        kind = self.pending_failures.pop(0)
        if kind == "unavailable":
            raise ERPError("unavailable", "503 Service unavailable", retryable=True, retry_after=1.0)
        if kind == "rate_limited":
            raise ERPError("rate_limited", "429 Too many requests", retryable=True, retry_after=2.0)
        if kind == "timeout":
            raise ERPError("timeout", "No answer after 5 seconds", retryable=True)
        return kind == "lost_response"

    # ---------- reads ----------

    def get_product(self, sku):
        self._check_failures()
        if sku not in self.products:
            raise ERPError("not_found", f"Product {sku} not found")
        return dict(self.products[sku])

    def find_products(self, category=None, max_stock=None):
        self._check_failures()
        found = []
        for product in self.products.values():
            if category is not None and product["category"] != category:
                continue
            if max_stock is not None and product["stock"] > max_stock:
                continue
            found.append(dict(product))
        return found

    def get_customer(self, customer_id):
        self._check_failures()
        if customer_id not in self.customers:
            raise ERPError("not_found", f"Customer {customer_id} not found")
        return dict(self.customers[customer_id])

    def get_invoice(self, invoice_id):
        self._check_failures()
        if invoice_id not in self.invoices:
            raise ERPError("not_found", f"Invoice {invoice_id} not found")
        return dict(self.invoices[invoice_id])

    # ---------- writes: applied once per idempotency key ----------

    def adjust_stock(self, sku, delta, reason, idempotency_key=None):
        lose_reply = self._check_failures()
        if idempotency_key in self.saved_replies:
            return {**self.saved_replies[idempotency_key], "replayed": True}

        if sku not in self.products:
            raise ERPError("not_found", f"Product {sku} not found")
        product = self.products[sku]
        if product["stock"] + delta < 0:
            raise ERPError("insufficient_stock", f"Stock is {product['stock']}, cannot apply {delta}")
        product["stock"] += delta
        self.movements.append({"sku": sku, "delta": delta, "reason": reason})
        reply = {"sku": sku, "stock": product["stock"]}

        if idempotency_key:
            self.saved_replies[idempotency_key] = reply
        if lose_reply:
            # The write is committed, but the client never gets the answer.
            raise ERPError("bad_gateway", "502 Connection reset. The write may have been applied.", retryable=True)
        return reply

    def create_refund(self, invoice_id, amount, reason, idempotency_key=None):
        lose_reply = self._check_failures()
        if idempotency_key in self.saved_replies:
            return {**self.saved_replies[idempotency_key], "replayed": True}

        if invoice_id not in self.invoices:
            raise ERPError("not_found", f"Invoice {invoice_id} not found")
        invoice = self.invoices[invoice_id]
        if invoice["status"] != "paid":
            raise ERPError("invalid_state", "Only paid invoices can be refunded")
        available = invoice["total"] - invoice["refunded"]
        if amount > available:
            raise ERPError("exceeds_refundable", f"Refundable balance is {available}")
        invoice["refunded"] += amount
        self.refunds.append({"invoice_id": invoice_id, "amount": amount, "reason": reason})
        reply = {"invoice_id": invoice_id, "refunded": amount, "remaining": available - amount}

        if idempotency_key:
            self.saved_replies[idempotency_key] = reply
        if lose_reply:
            raise ERPError("bad_gateway", "502 Connection reset. The write may have been applied.", retryable=True)
        return reply


erp = AcmeERP()
print(len(erp.products), "products,", len(erp.customers), "customers,", len(erp.invoices), "invoices")

In [ ]:
for product in erp.find_products(max_stock=10):
    print(product["sku"], product["name"], "stock:", product["stock"])

## 3. One tool call, by hand
The model never touches the ERP. It **proposes** a call as JSON, **our code** runs it and sends the result back.
Every agent is built on this exchange.

In [ ]:
tools = [{
    "type": "function",
    "function": {
        "name": "get_product",
        "description": "Get one product by SKU: name, price and stock.",
        "parameters": {"type": "object", "properties": {"sku": {"type": "string"}}, "required": ["sku"]},
    },
}]
messages = [
    {"role": "system", "content": "Answer questions about Acme inventory. Use the tools. Never guess numbers."},
    {"role": "user", "content": "How many units of SKU-007 do we have?"},
]

first = ask_model(messages, tools)
first.pop("tokens")
call = first["tool_calls"][0]
print("The model asks for:", call["function"]["name"], call["function"]["arguments"])

arguments = json.loads(call["function"]["arguments"])
result = erp.get_product(**arguments)            # our code decides to run it

messages.append(first)
messages.append({"role": "tool", "tool_call_id": call["id"], "content": json.dumps(result)})
second = ask_model(messages, tools)
print("Final answer:", second["content"])

That was one turn of a control loop, written out by hand. In Lab 1 we turn it into a real loop.